## How to run (English)

1. Download the dataset from Mendeley Data (doi:10.17632/3mbnb82mxd.2) and unzip it; you need the folder `Tomato Dataset/Dataset (raw)`.
2. **Google Colab (recommended, T4 GPU):** upload `Tomato Dataset` to your Google Drive (`MyDrive/Tomato Dataset`), then run
   `!git clone https://github.com/<user>/tomato-leaf /content/tomato-leaf` in a cell and open this notebook. Runtime → Change runtime type → T4 GPU.
3. **Locally:** set `TOMATO_DATA=/path/to/Tomato Dataset` (and optionally `TOMATO_REPO`, `TOMATO_OUT`) and start Jupyter from `notebooks/`.
4. Only cell **2) Paths** contains paths. Outputs go to `OUT_DIR/results_v7`; the repository's `results/` is only read.
5. This notebook retrains the 5 final PVTv2-B2 and Swin-T models (10 epochs each), saves fp16 weights (≈ 50 MB each) and computes Grad-CAM; it compares the new test predictions with the paper's `results/test_probs/`. Quick check: `TOMATO_SMOKE=1`.

Comments inside the code cells are in Turkish; variable names and outputs are in English.


# Domates yaprağı hastalığı — v7: PVTv2-B2 ve Swin-T final modellerinin kaydı + Grad-CAM

Bu notebook makaledeki sonuçları **değiştirmez**; çıktılar ayrı bir klasöre (`results_v7/`) yazılır.

1. PVTv2-B2 ve Swin-T, makaledeki final protokolle (598 trainval görüntüsü, ImageNet-1k başlangıcı, sabit E = CV en iyi epoch medyanı, seed 0–4) yeniden eğitilir.
2. Her final modelin ağırlıkları **fp16** olarak kaydedilir (`results_v7/weights/<model>__seed<k>.pt`, model başına ≈ 47–55 MB). Dosyada etiket, sınıflar, normalizasyon, epoch, seed, sürümler, test skoru ve SHA-256 bulunur. `load_model(ad, seed)` ile geri yüklenir.
3. Yeniden üretilebilirlik: yeni test skorları ve tahminler, makaledeki (`results/`) koşularla karşılaştırılır. GPU'daki deterministik olmayan işlemler nedeniyle küçük farklar beklenir; **makaledeki sayılar eski koşulardır**.
4. Grad-CAM (son aşama, 7×7): sınıf başına örnekler, hatalı tahminler, seed'ler arası kararlılık, iki model arası uyum ve silme (*deletion*) sadakat testi (CAM'e göre en önemli bölgeler maskelenince olasılık düşüşü ile rastgele maskeleme karşılaştırması).

Kaldığı yerden devam eder: ağırlık dosyası olan (model, seed) yeniden eğitilmez.

In [ ]:
# 0) Kurulum
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'timm>=1.0.9'], check=True)
import torch as _t; print('torch', _t.__version__, '| CUDA', _t.version.cuda, '| GPU', _t.cuda.get_device_name() if _t.cuda.is_available() else '-')

In [ ]:
# 1) Ayarlar
import os, json, time, math, glob, hashlib, random, shutil, warnings
import numpy as np, pandas as pd
SMOKE = bool(int(os.environ.get('TOMATO_SMOKE', '0')))   # yerel kısa deneme modu
MODELS = {'PVTv2-B2': ('pvt_v2_b2.in1k', 'C: ViT'),
          'Swin-T':   ('swin_tiny_patch4_window7_224.ms_in1k', 'C: ViT')}
CLASSES = ['Early blight', 'Late blight', 'Septoria leaf spot', 'Leaf mold', 'Yellow leaf curl virus', 'Healthy']
CLASSES_TR = ['Erken yanıklık', 'Geç yanıklık', 'Septoria yaprak lekesi', 'Yaprak küfü', 'TYLCV', 'Sağlıklı']
SEEDS = [0, 1, 2, 3, 4]
CFG = dict(img=224, cache_short=256, batch=32, max_epochs=40, patience=7, warmup=3,
           lr_backbone=1e-4, lr_head=1e-3, wd=0.05, label_smoothing=0.1, workers=2, amp=True)
PRETRAINED = True
SAVE_FP16 = True            # ağırlıkları yarı duyarlıkta kaydet (boyut yarıya iner)
EXPECTED_EPOCHS = {'PVTv2-B2': 10, 'Swin-T': 10}   # makaledeki değerler (CV en iyi epoch medyanı)
DEL_FRACS = [0.1, 0.2, 0.3, 0.5]  # silme testinde maskelenen alan oranları
N_RANDOM = 3                       # rastgele maskeleme tekrarı
if SMOKE:
    SEEDS = [0, 1]; CFG.update(batch=8, workers=0, amp=False); PRETRAINED = False
    DEL_FRACS = [0.2, 0.5]; N_RANDOM = 1
# AMP uyarısı: GradScaler ilk adımlarda inf gradyan nedeniyle optimizer adımını atlar; zamanlayıcı yine ilerler.
# Makaledeki protokolle birebir aynı kalması için sıra değiştirilmedi, yalnızca uyarı gizlendi.
warnings.filterwarnings('ignore', message=r'Detected call of `lr_scheduler\.step\(\)` before `optimizer\.step\(\)`')
print(list(MODELS), '| seed:', SEEDS, '| SMOKE =', SMOKE)

In [ ]:
# 2) Paths — the ONLY cell you need to edit
#   REPO_DIR : this repository (contains splits/ and results/)
#   DATA_ROOT: the Mendeley folder "Tomato Dataset" (must contain "Dataset (raw)")
#   OUT_DIR  : where NEW runs are written. Kept separate from the paper's results/ on purpose:
#              this notebook writes to OUT_DIR/results_v7 and only reads the paper's results/.
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR  = os.environ.get('TOMATO_REPO', '/content/tomato-leaf')     # !git clone https://github.com/<user>/tomato-leaf /content/tomato-leaf
    DATA_ROOT = os.environ.get('TOMATO_DATA', '/content/drive/MyDrive/Tomato Dataset')
    OUT_DIR   = os.environ.get('TOMATO_OUT',  '/content/drive/MyDrive/tomato-leaf-runs')
    CACHE = '/content/cache_raw'
else:
    REPO_DIR  = os.environ.get('TOMATO_REPO', os.path.abspath('..'))
    DATA_ROOT = os.environ['TOMATO_DATA']
    OUT_DIR   = os.environ.get('TOMATO_OUT', os.path.join(REPO_DIR, 'runs'))
    CACHE = os.path.join(OUT_DIR, '_cache')
assert os.path.isdir(os.path.join(DATA_ROOT, 'Dataset (raw)')), f"'Dataset (raw)' not found under DATA_ROOT = {DATA_ROOT}"
SPLIT_CSV = os.path.join(REPO_DIR, 'splits', 'raw_clean_group_split.csv')
assert os.path.exists(SPLIT_CSV), f'split file not found: {SPLIT_CSV}'
PROJ = OUT_DIR
RES = os.path.join(REPO_DIR, 'results')         # the paper's results (read-only: cv_runs.csv, final_test_runs.csv, test_probs/)
OUT = os.path.join(OUT_DIR, 'results_v7')       # ALL outputs of this notebook
WDIR, PDIR, GDIR = (os.path.join(OUT, x) for x in ('weights', 'test_probs', 'gradcam'))
for p in (OUT, WDIR, PDIR, GDIR, CACHE): os.makedirs(p, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT); print('PAPER RES =', RES); print('OUTPUT    =', OUT)


In [ ]:
# 3) Bölme dosyası + ham görüntülerin doğrulanması (MD5) + yerel önbellek (kısa kenar 256 px)
from PIL import Image
df = pd.read_csv(SPLIT_CSV)
assert len(df) == 748 and df.split.value_counts().to_dict() == {'trainval': 598, 'test': 150}
assert df.groupby('group').split.nunique().max() == 1, 'bir grup hem testte hem trainval’de!'
assert df[df.split == 'trainval'].groupby('group').fold.nunique().max() == 1, 'bir grup birden fazla katta!'

def _long(p): return ('\\\\?\\' + os.path.abspath(p)) if os.name == 'nt' else p
def cache_one(row):
    src = _long(os.path.join(DATA_ROOT, row.relpath))
    dst = os.path.join(CACHE, f'{row.Index:04d}.jpg')
    raw = open(src, 'rb').read()
    ok = hashlib.md5(raw).hexdigest() == row.md5
    if not os.path.exists(dst):
        im = Image.open(src).convert('RGB'); s = CFG['cache_short'] / min(im.size)
        im.resize((max(1, round(im.width * s)), max(1, round(im.height * s))), Image.BICUBIC).save(dst, quality=95)
    return dst, ok

from concurrent.futures import ThreadPoolExecutor
t0 = time.time()
with ThreadPoolExecutor(16) as ex: res = list(ex.map(cache_one, df.itertuples()))
df['path'] = [r[0] for r in res]; md5_ok = np.array([r[1] for r in res])
print(f'önbellek: {len(df)} görüntü, {time.time()-t0:.0f} s | MD5 eşleşmeyen: {(~md5_ok).sum()}')
assert md5_ok.all(), 'Drive’daki dosyalar denetlenen kopyayla aynı değil — durun ve kontrol edin.'

In [ ]:
# 4) Sızıntı kontrolü: test ile trainval arasında birebir (MD5) veya döndürme/çevirmeye dayanıklı pHash (d<=10) eşleşmesi olmamalı
from scipy.fft import dctn
def dihedral_phash(path):
    g = np.asarray(Image.open(path).convert('L').resize((32, 32), Image.BILINEAR), dtype=np.float32)
    out = []
    for k in range(4):
        for fl in (False, True):
            a = np.rot90(g, k); a = a[:, ::-1] if fl else a
            d = dctn(a, norm='ortho')[:8, :8].ravel(); out.append(np.packbits(d > np.median(d[1:])).view('>u8')[0])
    return out
H = np.array([dihedral_phash(p) for p in df.path], dtype=np.uint64)
LUT = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
te, tv = np.where(df.split == 'test')[0], np.where(df.split == 'trainval')[0]
dmin = LUT[(H[te, 0][:, None, None] ^ H[tv][None]).view(np.uint8).reshape(len(te), len(tv), 8, 8)].sum(-1).min(-1)
n_md5 = len(set(df.md5.iloc[te]) & set(df.md5.iloc[tv]))
print(f'test↔trainval: ortak MD5 = {n_md5} | en küçük pHash mesafesi = {dmin.min()} (eşik 10)')
assert n_md5 == 0 and dmin.min() > 10, 'SIZINTI: eğitim başlatılmadı.'

In [ ]:
# 5) Veri yükleyiciler — augmentasyon YALNIZCA train'de
import torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('cihaz:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '')
IMGS = {p: Image.open(p).convert('RGB') for p in df.path}          # 748 küçük görüntü RAM'de

def tfms(mean, std, train):
    s = CFG['img']
    if train:
        return T.Compose([T.RandomResizedCrop(s, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                          T.RandomRotation(20), T.ColorJitter(0.2, 0.2, 0.2, 0.02), T.ToTensor(), T.Normalize(mean, std)])
    return T.Compose([T.Resize(s), T.CenterCrop(s), T.ToTensor(), T.Normalize(mean, std)])

class DS(Dataset):
    def __init__(self, sub, tf): self.p, self.y, self.tf = sub.path.tolist(), sub.label.tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(IMGS[self.p[i]]), self.y[i]

def loaders(train_df, eval_df, mean, std, seed):
    g = torch.Generator().manual_seed(seed)
    tl = DataLoader(DS(train_df, tfms(mean, std, True)), CFG['batch'], shuffle=True, drop_last=len(train_df) > CFG['batch'],
                    num_workers=CFG['workers'], generator=g, pin_memory=DEV == 'cuda', persistent_workers=CFG['workers'] > 0)
    el = None if eval_df is None else DataLoader(DS(eval_df, tfms(mean, std, False)), 64, num_workers=CFG['workers'])
    return tl, el

In [ ]:
# 6) Model, eğitim ve değerlendirme (makaledeki train_one ile aynı)
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True
def build(tag, pretrained=None):
    m = timm.create_model(tag, pretrained=PRETRAINED if pretrained is None else pretrained, num_classes=len(CLASSES))
    cfg = m.pretrained_cfg; return m.to(DEV), cfg.get('mean', (0.485, 0.456, 0.406)), cfg.get('std', (0.229, 0.224, 0.225))
def metrics(y, p):
    return dict(macro_f1=f1_score(y, p, average='macro'), acc=accuracy_score(y, p), bal_acc=balanced_accuracy_score(y, p),
                mcc=matthews_corrcoef(y, p), kappa=cohen_kappa_score(y, p))
@torch.no_grad()
def predict(m, dl):
    m.eval(); P = []
    for x, _ in dl:
        with torch.autocast(DEV, enabled=CFG['amp'] and DEV == 'cuda'): P.append(m(x.to(DEV, non_blocking=True)).float().softmax(1).cpu())
    return torch.cat(P).numpy()
def train_one(tag, train_df, val_df, seed, epochs=None):
    set_seed(seed); m, mean, std = build(tag); tl, vl = loaders(train_df, val_df, mean, std, seed)
    head = set(id(p) for p in m.get_classifier().parameters())
    opt = torch.optim.AdamW([{'params': [p for p in m.parameters() if id(p) not in head], 'lr': CFG['lr_backbone']},
                             {'params': [p for p in m.parameters() if id(p) in head], 'lr': CFG['lr_head']}], weight_decay=CFG['wd'])
    E = epochs or CFG['max_epochs']; steps = E * len(tl); wsteps = CFG['warmup'] * len(tl)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / wsteps if s < wsteps else 0.5 * (1 + math.cos(math.pi * (s - wsteps) / max(1, steps - wsteps))))
    cnt = np.bincount(train_df.label, minlength=len(CLASSES)); w = torch.tensor(cnt.sum() / (len(CLASSES) * np.maximum(cnt, 1)), dtype=torch.float32, device=DEV)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG['label_smoothing'])
    scaler = torch.amp.GradScaler(enabled=CFG['amp'] and DEV == 'cuda')
    for ep in range(1, E + 1):
        m.train()
        for x, y in tl:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.autocast(DEV, enabled=CFG['amp'] and DEV == 'cuda'): loss = crit(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
    return m, mean, std

def safe(name): return name.replace('/', '-')
def wpath(name, seed): return os.path.join(WDIR, f'{safe(name)}__seed{seed}.pt')
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def load_model(name, seed, device=None):
    # Kaydedilmiş final modeli geri yükler (fp16 → fp32). Dönen: model (eval), mean, std, meta
    ck = torch.load(wpath(name, seed), map_location='cpu', weights_only=False)
    meta = ck['meta']; m = timm.create_model(meta['timm_tag'], pretrained=False, num_classes=len(meta['classes']))
    m.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in ck['state_dict'].items()})
    return m.to(device or DEV).eval(), tuple(meta['mean']), tuple(meta['std']), meta

In [ ]:
# 7) Final epoch sayıları: makaledeki CV koşularından (en iyi epoch medyanı, yuvarlanmış)
CV_CSV = os.path.join(RES, 'cv_runs.csv')
if os.path.exists(CV_CSV):
    cv = pd.read_csv(CV_CSV)
    EPOCHS = cv[cv.model.isin(MODELS)].groupby('model').best_epoch.median().round().astype(int).clip(lower=1).to_dict()
else:
    print('UYARI: cv_runs.csv yok → makaledeki değerler kullanılıyor'); EPOCHS = dict(EXPECTED_EPOCHS)
if SMOKE: EPOCHS = {k: 1 for k in MODELS}
print('epoch:', EPOCHS)
if not SMOKE:
    assert EPOCHS == EXPECTED_EPOCHS, f'epoch sayıları makaledekiyle aynı değil: {EPOCHS}'
TE = df[df.split == 'test'].reset_index(drop=True); TVall = df[df.split == 'trainval'].reset_index(drop=True)
if SMOKE: TVall, TE = TVall.groupby('label').head(4).reset_index(drop=True), TE.groupby('label').head(2).reset_index(drop=True)
y_te = TE.label.values
print(f'final eğitim: {len(TVall)} görüntü | test: {len(TE)} görüntü')

In [ ]:
# 8) Final eğitim + ağırlık kaydı (fp16) + geri yükleme kontrolü
FIN_CSV = os.path.join(OUT, 'final_test_runs_v7.csv')
fin = pd.read_csv(FIN_CSV) if os.path.exists(FIN_CSV) else pd.DataFrame()
done = set(zip(fin.model, fin.seed)) if len(fin) else set()
import timm as _timm
for name, (tag, grp) in MODELS.items():
    for seed in SEEDS:
        if (name, seed) in done and os.path.exists(wpath(name, seed)):
            print(f'{name} seed{seed}: zaten var, atlanıyor'); continue
        t0 = time.time(); m, mean, std = train_one(tag, TVall, None, seed, epochs=EPOCHS[name]); mins = (time.time() - t0) / 60
        _, tl = loaders(TVall, TE, mean, std, seed)
        P32 = predict(m, tl)                                     # eğitim sonrası bellekteki fp32 model
        sd = {k: (v.detach().half() if (SAVE_FP16 and v.is_floating_point()) else v.detach()).cpu() for k, v in m.state_dict().items()}
        met32 = metrics(y_te, P32.argmax(1))
        meta = dict(model=name, timm_tag=tag, group=grp, seed=seed, epochs=EPOCHS[name], classes=CLASSES, mean=list(mean), std=list(std),
                    img=CFG['img'], dtype='fp16' if SAVE_FP16 else 'fp32', cfg=CFG, n_train=len(TVall),
                    torch=torch.__version__, timm=_timm.__version__, test_macro_f1_fp32=float(met32['macro_f1']),
                    created=time.strftime('%Y-%m-%d %H:%M:%S'))
        tmp = wpath(name, seed) + '.tmp'; torch.save({'state_dict': sd, 'meta': meta}, tmp); os.replace(tmp, wpath(name, seed))
        del m; torch.cuda.empty_cache() if DEV == 'cuda' else None
        m2, _, _, _ = load_model(name, seed); P = predict(m2, tl); del m2       # kaydedilen dosyadan geri yüklenen model
        np.save(os.path.join(PDIR, f'{safe(name)}__seed{seed}.npy'), P)
        row = dict(model=name, seed=seed, epochs=EPOCHS[name], minutes=round(mins, 2), **metrics(y_te, P.argmax(1)),
                   macro_f1_fp32=met32['macro_f1'], fp16_pred_agree=float((P.argmax(1) == P32.argmax(1)).mean()),
                   fp16_max_abs_prob_diff=float(np.abs(P - P32).max()), file_MB=round(os.path.getsize(wpath(name, seed)) / 1e6, 1),
                   sha256=sha256(wpath(name, seed)))
        pd.DataFrame([row]).to_csv(FIN_CSV, mode='a', header=not os.path.exists(FIN_CSV), index=False)
        print(f"{name:9s} seed{seed} | {mins:.1f} dk | TEST macroF1 = {row['macro_f1']:.4f} (fp32 {met32['macro_f1']:.4f}) | "
              f"fp16 uyum {row['fp16_pred_agree']:.3f} | {row['file_MB']} MB")
fin = pd.read_csv(FIN_CSV); print(fin[['model', 'seed', 'macro_f1', 'fp16_pred_agree', 'file_MB']].to_string(index=False))

In [ ]:
# 9) Yeniden üretilebilirlik: yeni koşular (v7) ile makaledeki koşuların karşılaştırılması
old_fin = pd.read_csv(os.path.join(RES, 'final_test_runs.csv')) if os.path.exists(os.path.join(RES, 'final_test_runs.csv')) else None
def ens(name, d):
    fs = [os.path.join(d, f'{safe(name)}__seed{s}.npy') for s in SEEDS]
    return np.mean([np.load(f) for f in fs], 0) if all(os.path.exists(f) for f in fs) else None
rep = []
for name in MODELS:
    new = fin[fin.model == name].sort_values('seed'); r = dict(model=name, v7_seed_mean=new.macro_f1.mean(), v7_seed_sd=new.macro_f1.std())
    En = ens(name, PDIR); r['v7_ens_macro_f1'] = f1_score(y_te, En.argmax(1), average='macro')
    if old_fin is not None and not SMOKE:
        o = old_fin[old_fin.model == name]; r.update(paper_seed_mean=o.macro_f1.mean(), paper_seed_sd=o.macro_f1.std())
        Eo = ens(name, os.path.join(RES, 'test_probs'))
        if Eo is not None:
            r.update(paper_ens_macro_f1=f1_score(y_te, Eo.argmax(1), average='macro'), ens_pred_agree=float((Eo.argmax(1) == En.argmax(1)).mean()))
            r['per_seed_pred_agree'] = np.mean([(np.load(os.path.join(RES, 'test_probs', f'{safe(name)}__seed{s}.npy')).argmax(1) ==
                                                 np.load(os.path.join(PDIR, f'{safe(name)}__seed{s}.npy')).argmax(1)).mean() for s in SEEDS])
    rep.append(r)
rep = pd.DataFrame(rep); rep.to_csv(os.path.join(OUT, 'reproducibility_v7.csv'), index=False); print(rep.round(4).to_string(index=False))

In [ ]:
# 10) Grad-CAM: son aşama çıktısı (PVTv2-B2: stages[-1], NCHW 7×7×512; Swin-T: layers[-1], NHWC 7×7×768)
import torch.nn.functional as F
def target_layer(m, name): return m.stages[-1] if name.startswith('PVT') else m.layers[-1]
def to_nchw(a):   # Swin NHWC (B,H,W,C) → NCHW; PVTv2 zaten NCHW
    return a.permute(0, 3, 1, 2) if (a.ndim == 4 and a.shape[1] == a.shape[2] and a.shape[3] != a.shape[1]) else a

def gradcam(m, name, x, targets):
    # x: (B,3,224,224) normalize edilmiş; targets: (B,) sınıf indisleri → (B,h,w) ham CAM (ReLU, normalize edilmemiş), logits
    st = {}
    def fh(_, __, out):
        st['a'] = out; out.register_hook(lambda g: st.__setitem__('g', g))
    h = target_layer(m, name).register_forward_hook(fh)
    try:
        m.zero_grad(set_to_none=True); x = x.clone().requires_grad_(True)
        with torch.autocast(DEV, enabled=False): logits = m(x.float())
        logits.gather(1, targets[:, None]).sum().backward()
        A, G = to_nchw(st['a']).detach().float(), to_nchw(st['g']).detach().float()
        cam = F.relu((G.mean((2, 3), keepdim=True) * A).sum(1))
    finally: h.remove()
    return cam, logits.detach().float().softmax(1)

def up(cam, size=224):   # (B,h,w) → (B,size,size), görüntü başına [0,1]
    c = F.interpolate(cam[:, None], size=(size, size), mode='bilinear', align_corners=False)[:, 0]
    c = c - c.flatten(1).min(1)[0][:, None, None]; return c / (c.flatten(1).max(1)[0][:, None, None] + 1e-8)

def test_tensor(mean, std):
    tf = tfms(mean, std, False); return torch.stack([tf(IMGS[p]) for p in TE.path])
DISP = np.stack([np.asarray(T.CenterCrop(CFG['img'])(T.Resize(CFG['img'])(IMGS[p]))) for p in TE.path])   # gösterim için

ENSP, CAM_PRED, CAM_TRUE, SEEDP = {}, {}, {}, {}
yt = torch.tensor(y_te)
for name in MODELS:
    probs = [np.load(os.path.join(PDIR, f'{safe(name)}__seed{s}.npy')) for s in SEEDS]
    ENSP[name] = np.mean(probs, 0); pred_ens = torch.tensor(ENSP[name].argmax(1))
    cp, ctr = [], []
    for s in SEEDS:
        m, mean, std, _ = load_model(name, s); X = test_tensor(mean, std); a, b = [], []
        for i in range(0, len(X), 16):
            xb = X[i:i+16].to(DEV)
            a.append(gradcam(m, name, xb, pred_ens[i:i+16].to(DEV))[0].cpu()); b.append(gradcam(m, name, xb, yt[i:i+16].to(DEV))[0].cpu())
        cp.append(torch.cat(a)); ctr.append(torch.cat(b)); del m; torch.cuda.empty_cache() if DEV == 'cuda' else None
    CAM_PRED[name], CAM_TRUE[name] = torch.stack(cp), torch.stack(ctr)     # (seed, N, 7, 7)
    print(name, 'CAM boyutu:', tuple(CAM_PRED[name].shape))
np.savez_compressed(os.path.join(GDIR, 'gradcam_raw_7x7.npz'), **{f'{safe(n)}__{k}': v.numpy() for n in MODELS
                    for k, v in (('pred', CAM_PRED[n]), ('true', CAM_TRUE[n]))}, y_true=y_te, **{f'{safe(n)}__ens_probs': ENSP[n] for n in MODELS})
def ens_cam(name, which='pred', size=224):   # seed'ler arası ortalama (her seed önce [0,1]'e normalize)
    C = (CAM_PRED if which == 'pred' else CAM_TRUE)[name]; return torch.stack([up(c, size) for c in C]).mean(0)

In [ ]:
# 11) Nicel Grad-CAM analizleri: seed kararlılığı, modeller arası uyum, silme (deletion) sadakat testi
from scipy import stats
from itertools import combinations
def pear(a, b):   # görüntü başına Pearson r, a,b: (N,H,W)
    a = a.flatten(1) - a.flatten(1).mean(1, keepdim=True); b = b.flatten(1) - b.flatten(1).mean(1, keepdim=True)
    return ((a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-8)).numpy()
def topk_iou(a, b, q=0.2):
    qa = torch.quantile(a.flatten(1), 1 - q, dim=1)[:, None, None]; qb = torch.quantile(b.flatten(1), 1 - q, dim=1)[:, None, None]
    A, B = a >= qa, b >= qb; return ((A & B).flatten(1).sum(1) / ((A | B).flatten(1).sum(1) + 1e-8)).numpy()

per = pd.DataFrame(dict(idx=np.arange(len(TE)), relpath=TE.relpath, y_true=y_te))
summary = {}
for name in MODELS:
    U = torch.stack([up(c, 56) for c in CAM_PRED[name]])                       # (seed, N, 56, 56)
    pr = np.mean([pear(U[i], U[j]) for i, j in combinations(range(len(SEEDS)), 2)], 0)
    per[f'{name}_pred'] = ENSP[name].argmax(1); per[f'{name}_p_true'] = ENSP[name][np.arange(len(TE)), y_te]
    per[f'{name}_seed_r'] = pr
    summary[name] = dict(seed_pairwise_r_mean=float(pr.mean()), seed_pairwise_r_sd=float(pr.std(ddof=1)),
                         ens_macro_f1=float(f1_score(y_te, ENSP[name].argmax(1), average='macro')))
a, b = ens_cam('PVTv2-B2', 'true', 56), ens_cam('Swin-T', 'true', 56)
per['cross_model_r'] = pear(a, b); per['cross_model_iou20'] = topk_iou(a, b)
summary['cross_model'] = dict(r_mean=float(per.cross_model_r.mean()), r_sd=float(per.cross_model_r.std()),
                              iou20_mean=float(per.cross_model_iou20.mean()), iou20_sd=float(per.cross_model_iou20.std()),
                              iou20_chance=0.2 / (2 - 0.2))
# --- Silme testi: CAM'e göre en önemli %f alan bulanık görüntüyle değiştirilir; aynı alan rastgele (16×16 blok) seçilince ile karşılaştırılır.
blur = T.GaussianBlur(31, sigma=10.0); rng = torch.Generator().manual_seed(0); DEL = []
for name in MODELS:
    tgt = torch.tensor(ENSP[name].argmax(1)); S = ens_cam(name, 'pred', CFG['img'])           # hedef: topluluğun tahmin ettiği sınıf
    R = [F.interpolate(torch.rand(len(TE), 1, 14, 14, generator=rng), size=CFG['img'], mode='nearest')[:, 0] for _ in range(N_RANDOM)]
    masks = {}
    for f in DEL_FRACS:
        masks[('cam', f, 0)] = S >= torch.quantile(S.flatten(1), 1 - f, dim=1)[:, None, None]
        for r in range(N_RANDOM): masks[('rand', f, r)] = R[r] >= torch.quantile(R[r].flatten(1), 1 - f, dim=1)[:, None, None]
    acc = {k: np.zeros(len(TE)) for k in masks}; base = np.zeros(len(TE))
    for s in SEEDS:
        m, mean, std, _ = load_model(name, s); X = test_tensor(mean, std); Xb = blur(X)
        with torch.no_grad():
            def P(x): return torch.cat([m(x[i:i+64].to(DEV)).float().softmax(1).cpu() for i in range(0, len(x), 64)])[torch.arange(len(x)), tgt].numpy()
            base += P(X) / len(SEEDS)
            for k, M in masks.items():
                Mk = M[:, None].float(); acc[k] += P(X * (1 - Mk) + Xb * Mk) / len(SEEDS)
        del m; torch.cuda.empty_cache() if DEV == 'cuda' else None
    for f in DEL_FRACS:
        d_cam = base - acc[('cam', f, 0)]; d_rnd = base - np.mean([acc[('rand', f, r)] for r in range(N_RANDOM)], 0)
        w = stats.wilcoxon(d_cam, d_rnd) if len(TE) > 5 and np.any(d_cam != d_rnd) else None
        DEL.append(dict(model=name, frac=f, base_prob=base.mean(), drop_cam=d_cam.mean(), drop_cam_sd=d_cam.std(ddof=1),
                        drop_random=d_rnd.mean(), drop_random_sd=d_rnd.std(ddof=1), wilcoxon_p=(w.pvalue if w else np.nan), n=len(TE)))
DEL = pd.DataFrame(DEL); DEL.to_csv(os.path.join(GDIR, 'gradcam_deletion.csv'), index=False)
per.to_csv(os.path.join(GDIR, 'gradcam_per_image.csv'), index=False)
summary['deletion'] = DEL.to_dict('records'); summary['settings'] = dict(target_layers={'PVTv2-B2': 'stages[-1] (7x7x512)', 'Swin-T': 'layers[-1] (7x7x768)'},
    cam_target='topluluk tahmini (pred) / gerçek sınıf (true)', seeds=SEEDS, del_fracs=DEL_FRACS, n_random=N_RANDOM, blur='GaussianBlur(31, sigma=10)')
json.dump(summary, open(os.path.join(GDIR, 'gradcam_summary.json'), 'w'), indent=1, ensure_ascii=False, default=float)
print(json.dumps({k: v for k, v in summary.items() if k != 'deletion'}, indent=1, ensure_ascii=False, default=float))
print(DEL.round(4).to_string(index=False))

In [ ]:
# 12) Grad-CAM figürleri (Türkçe, 300 dpi)
import matplotlib; import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 8, 'axes.titlesize': 8, 'savefig.dpi': 300, 'font.family': 'DejaVu Sans'})
def overlay(ax, i, cam=None, title=None):
    ax.imshow(DISP[i]);
    if cam is not None: ax.imshow(cam, cmap='jet', alpha=0.45, vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([])
    if title: ax.set_title(title, fontsize=7)
EP, ES = ens_cam('PVTv2-B2', 'pred'), ens_cam('Swin-T', 'pred'); ETP, ETS = ens_cam('PVTv2-B2', 'true'), ens_cam('Swin-T', 'true')
pp, ps = ENSP['PVTv2-B2'].argmax(1), ENSP['Swin-T'].argmax(1)

# (a) Sınıf başına 2 örnek: iki modelin de doğru sınıfladığı görüntülerden RASTGELE (seed 0) seçilmiştir — en iyi örnekler seçilmedi.
r = np.random.default_rng(0); K = 2
fig, axs = plt.subplots(len(CLASSES), 3 * K, figsize=(7.2, 1.25 * len(CLASSES)))
for c in range(len(CLASSES)):
    ok = np.where((y_te == c) & (pp == c) & (ps == c))[0]; pick = r.choice(ok, min(K, len(ok)), replace=False) if len(ok) else []
    for j in range(K):
        ax3 = axs[c, 3*j:3*j+3]
        if j >= len(pick):
            for a_ in ax3: a_.axis('off')
            continue
        i = pick[j]
        overlay(ax3[0], i, None, 'Görüntü' if c == 0 else None); overlay(ax3[1], i, EP[i].numpy(), 'PVTv2-B2' if c == 0 else None)
        overlay(ax3[2], i, ES[i].numpy(), 'Swin-T' if c == 0 else None)
    axs[c, 0].set_ylabel(CLASSES_TR[c], fontsize=7)
fig.tight_layout(pad=0.3); fig.savefig(os.path.join(GDIR, 'sekil_gradcam_siniflar.png'), bbox_inches='tight'); plt.show()

# (b) PVTv2-B2 topluluğunun TÜM hatalı tahminleri: tahmin edilen ve gerçek sınıf için CAM; Swin-T'nin aynı görüntüdeki tahmini
err = np.where(pp != y_te)[0]
if len(err):
    fig, axs = plt.subplots(len(err), 4, figsize=(5.6, 1.4 * len(err)), squeeze=False)
    for k, i in enumerate(err):
        overlay(axs[k, 0], i, None, None); axs[k, 0].set_ylabel(f'Gerçek: {CLASSES_TR[y_te[i]]}', fontsize=6)
        overlay(axs[k, 1], i, EP[i].numpy(), f'PVTv2 → {CLASSES_TR[pp[i]]}')
        overlay(axs[k, 2], i, ETP[i].numpy(), f'PVTv2, gerçek sınıf')
        overlay(axs[k, 3], i, ES[i].numpy(), f'Swin-T → {CLASSES_TR[ps[i]]}')
    fig.tight_layout(pad=0.3); fig.savefig(os.path.join(GDIR, 'sekil_gradcam_hatalar.png'), bbox_inches='tight'); plt.show()
print(f'PVTv2-B2 topluluk hatası: {len(err)} | bunlardan Swin-T doğru: {int((ps[err] == y_te[err]).sum())}')

# (c) Seed kararlılığı örneği: seed-arası uyumu medyana en yakın görüntü, 5 seed ayrı ayrı
fig, axs = plt.subplots(2, len(SEEDS) + 1, figsize=(7.2, 2.6), squeeze=False)
for row, name in enumerate(MODELS):
    i = int(np.argmin(np.abs(per[f'{name}_seed_r'] - per[f'{name}_seed_r'].median())))
    overlay(axs[row, 0], i, None, None); axs[row, 0].set_ylabel(name, fontsize=7)
    for j, s in enumerate(SEEDS): overlay(axs[row, j + 1], i, up(CAM_PRED[name][j, i:i+1])[0].numpy(), f'seed {s}')
fig.tight_layout(pad=0.3); fig.savefig(os.path.join(GDIR, 'sekil_gradcam_seedler.png'), bbox_inches='tight'); plt.show()

# (d) Silme eğrileri
fig, ax = plt.subplots(figsize=(3.5, 2.6))
for name, col in zip(MODELS, ['#1f77b4', '#d62728']):
    d = DEL[DEL.model == name]; base = d.base_prob.iloc[0]; x = [0] + list(d.frac)
    ax.plot(x, [base] + list(base - d.drop_cam), '-o', color=col, ms=3, label=f'{name} – Grad-CAM')
    ax.plot(x, [base] + list(base - d.drop_random), '--s', color=col, ms=3, alpha=0.7, label=f'{name} – rastgele')
ax.set_xlabel('Maskelenen alan oranı'); ax.set_ylabel('Tahmin edilen sınıfın olasılığı (ortalama)')
ax.legend(fontsize=6, frameon=False); ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(GDIR, 'sekil_gradcam_silme.png'), bbox_inches='tight'); plt.show()
print('Çıktılar:', sorted(os.listdir(GDIR)))

In [ ]:
# 13) Özet ve kayıt listesi
print(pd.read_csv(os.path.join(OUT, 'reproducibility_v7.csv')).round(4).to_string(index=False))
tot = sum(os.path.getsize(p) for p in glob.glob(os.path.join(WDIR, '*.pt'))) / 1e6
print(f'\nAğırlık dosyaları: {len(glob.glob(os.path.join(WDIR, "*.pt")))} adet, toplam {tot:.0f} MB → {WDIR}')
print('Geri yükleme örneği: m, mean, std, meta = load_model("PVTv2-B2", 0)')